# LangGraph Module · Day 01 — LangChain Core Concepts

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2 hrs

### Today's tasks (from the plan)
- [ ] Understand the core ideas: **chains, runnables, LCEL**
- [ ] Build a basic **LCEL chain**: `prompt + model + parser`
- [ ] Run the chain locally against a real LLM (**Google Gemini**)

### The one big idea
LangGraph (the agent framework you'll live in) is built **on top of** LangChain's core. Before graphs, master the brick they're all made of: the **Runnable**, and the **`|` pipe** that snaps Runnables together (**LCEL**). Get this, and every LangGraph node later feels familiar.

> Plain English: a *chain* = prompt → model → parser, wired with `|`. Each piece is a *Runnable*. LCEL is the wiring language.


## 0. Setup — load the API key

This notebook runs against **Google Gemini**. It needs `GOOGLE_API_KEY` in a `.env` file at the project root:
```
GOOGLE_API_KEY=your-key-here
```
`load_dotenv()` reads that file into environment variables so the model can authenticate.

Install (if needed):
```bash
pip install langchain langchain-core langchain-google-genai python-dotenv
```


In [1]:
from dotenv import load_dotenv
import os, langchain_core, langchain

load_dotenv()   # reads .env -> environment

print("langchain-core:", langchain_core.__version__)
print("langchain     :", langchain.__version__)
print("GOOGLE_API_KEY:", "loaded ✅" if os.getenv("GOOGLE_API_KEY") else "MISSING ❌ — add it to .env")


langchain-core: 1.4.8
langchain     : 1.3.11
GOOGLE_API_KEY: loaded ✅


## 1. The `Runnable` — the single most important concept

**Problem it solves:** LangChain has many component types — prompt templates, models, parsers, retrievers, custom functions. If each had its own method names (`prompt.format()`, `model.predict()`, `parser.parse()`), wiring them would be glue-code spaghetti.

**The fix:** every component implements the **same interface**, the `Runnable`. They *all* share these methods:

| Method | Plain English | When |
|--------|---------------|------|
| `.invoke(x)` | run once, input → output | normal call |
| `.batch([x1,x2])` | run many inputs efficiently | bulk jobs |
| `.stream(x)` | yield output in pieces as it's made | live token UI |
| `.ainvoke / .abatch / .astream` | the `async` versions | inside async agents |

Because every piece speaks the same verbs, you can **plug any piece into any other** without adapters. That uniformity is the whole foundation.


In [2]:
from langchain_core.runnables import RunnableLambda

# A plain Python function becomes a Runnable by wrapping it.
# Now this ordinary function speaks invoke / batch / stream like everything else.
response = RunnableLambda(lambda text: text.upper() + "!")

print("invoke:", response.invoke("hello"))
print("batch :", response.batch(["hi", "there"]))


invoke: HELLO!
batch : ['HI!', 'THERE!']


☝️ Nothing special about that function — wrapping it as a `Runnable` gives it `invoke`/`batch`/`stream` for free. Prompts, models, and parsers are *also* Runnables, so they behave the same way.

## 2. LCEL & the `|` pipe — wiring Runnables together

**LCEL** = *LangChain Expression Language*. Not a new language — one operator: the pipe **`|`**.

`a | b` means: *"run `a`, then feed its output into `b`."* Like a Unix shell pipe (`cat file | grep x`). The result is itself a Runnable, so you keep piping.

**Why `|` instead of nesting function calls?**


In [3]:
from langchain_core.runnables import RunnableLambda

add_exclaim = RunnableLambda(lambda s: s + "!")
to_upper    = RunnableLambda(lambda s: s.upper())
add_star    = RunnableLambda(lambda s: "* " + s + " *")

# Without LCEL: read inside-out, order reversed from reading
response = add_star.invoke(to_upper.invoke(add_exclaim.invoke("hi")))
print("response :", response)

# With LCEL: reads left-to-right in run order. Self-documenting.
pipeline_response = add_exclaim | to_upper | add_star
print("pipeline_response  :", pipeline_response.invoke("hi"))


response : * HI! *
pipeline_response  : * HI! *


Same result, but `add_exclaim | to_upper | add_star` reads in the **order things happen**. For a multi-step agent, that readability is the difference between maintainable and not. And `pipeline` is itself a Runnable, so it gets `.batch()` free:

In [4]:
print("batch:", pipeline_response.batch(["hi", "yo"]))


batch: ['* HI! *', '* YO! *']


## 3. Piece 1 — the Prompt Template

**Problem:** an LLM call needs a fully-written prompt. Hardcoding (`f"Answer this: {q}"`) mixes the *template* with the *values*, can't be reused, and gets messy with system/user roles.

**`ChatPromptTemplate`** separates the **fixed template** from the **runtime values**. Write placeholders (`{role}`, `{question}`) once; fill them per call. It also models chat **roles** (system / human / ai) properly.

Its job in the chain: **dict of values → formatted prompt (messages)**.


In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in one short sentence."),
    ("human", "{question}"),
])

# Feed a dict -> get formatted chat messages (system + human)
filled = prompt.invoke({"role": "finance assistant", "question": "What is LCEL?"})
for m in filled.messages:
    print(f"[{m.type}] {m.content}")


[system] You are a finance assistant. Answer in one short sentence.
[human] What is LCEL?


It's a **Runnable** too — same `.invoke()`. Input = dict, output = messages ready for a model.

## 4. Piece 2 — the Model (Google Gemini)

The model takes formatted messages and returns an **`AIMessage`** — a message *object*, not a plain string (that detail matters for Piece 3).

`ChatGoogleGenerativeAI` is the Runnable wrapper around Gemini. `temperature=0` makes output deterministic — good for repeatable study.


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

answer_msg = model.invoke(filled)        # messages in -> AIMessage out
print("type :", type(answer_msg).__name__)
print("text :", answer_msg.content)      # note: real content lives in .content


type : AIMessage
text : LCEL (LangChain Expression Language) is a framework for building applications with large language models (LLMs) using LangChain.


The return is an `AIMessage`, and the text sits in `.content`. Reaching into `.content` by hand everywhere is brittle — that's what the parser fixes next.

## 5. Piece 3 — the Output Parser

**Problem:** the model returned an `AIMessage`, but code usually wants the **plain text** inside. And later you'll want richer parsing (JSON → dict, lists, Pydantic).

**`StrOutputParser`** does the simplest version: **`AIMessage` → its text string**. Standard last link so the chain outputs a clean `str`.

Its job: **model message → usable Python value**.


In [7]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
text = parser.invoke(answer_msg)      # AIMessage -> str
print("type :", type(text).__name__)
print("value:", text)


type : TextAccessor
value: LCEL (LangChain Expression Language) is a framework for building applications with large language models (LLMs) using LangChain.


## 6. Put it together — `prompt | model | parser`

Today's deliverable. One readable line wires all three. Data flows:

```
dict  ─▶  prompt  ─▶  messages  ─▶  model  ─▶  AIMessage  ─▶  parser  ─▶  str
```

Each arrow is a `|`. Each box is a Runnable.


In [8]:
chain = prompt | model | parser    # the whole LCEL chain, one Runnable

result = chain.invoke({"role": "finance assistant", "question": "What is LCEL?"})
print(type(result).__name__, "->", result)


TextAccessor -> LCEL (LangChain Expression Language) is a declarative way to compose chains in LangChain, enabling more flexible and efficient AI application development.


That single `chain` now has the **full Runnable toolkit** for free — no extra code:

In [9]:
# batch: many inputs in one call
inputs = [
    {"role": "tutor", "question": "Explain runnables in one line"},
    {"role": "tutor", "question": "Explain the LCEL pipe in one line"},
]
for out in chain.batch(inputs):
    print("-", out)


- Runnables are objects that define a task or a piece of code to be executed.
- The LCEL pipe (`|`) chains runnables together, allowing the output of one to become the input of the next in a sequence.


In [10]:
# stream: receive output in chunks as it is produced (drives live "typing" UIs)
for chunk in chain.stream({"role": "tutor", "question": "Say hello in 3 words"}):
    print(repr(chunk))


'Hello there!'
''


**Why this is the whole point:** you wrote `prompt | model | parser` once. `invoke`, `batch`, `stream`, and their async twins came along automatically because each piece — and the chain itself — is a Runnable. *That* is what LCEL buys you.

## 7. Adding your own steps — `RunnableLambda` & `RunnablePassthrough`

Real chains need custom logic between standard pieces (clean input, post-process output). Two helpers:

- **`RunnableLambda(fn)`** — turn any function into a chain step.
- **`RunnablePassthrough`** — pass input through unchanged, or **add** computed fields alongside it (handy when the prompt needs several keys).


In [11]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Pre-step: normalise the question before it hits the prompt.
clean = RunnableLambda(lambda d: {**d, "question": d["question"].strip().capitalize()})

# Post-step: tag the answer.
label = RunnableLambda(lambda s: f"ANSWER: {s}")

full = clean | prompt | model | parser | label
print(full.invoke({"role": "assistant", "question": "  what is lcel?"}))


ANSWER: LCEL stands for LangChain Expression Language.


In [12]:
# RunnablePassthrough.assign: keep original input AND add a derived field.
add_len = RunnablePassthrough.assign(q_length=lambda d: len(d["question"]))
print(add_len.invoke({"question": "hello world"}))   # original kept, q_length added


{'question': 'hello world', 'q_length': 11}


## 8. Why LCEL instead of a plain Python function?

You *could* skip all this and write a normal function. Compare honestly:


In [13]:
# --- Plain Python: works, but you hand-build everything ---
def plain_chain(values: dict) -> str:
    msgs = prompt.invoke(values)       # still need the prompt
    ai = model.invoke(msgs)            # the model call
    return ai.content                  # manual .content extraction

print(plain_chain({"role": "finance assistant", "question": "Define LCEL in one line"}))
# To get batching: write a for-loop. Streaming: rewrite with yield.
# Async: rewrite again with async/await. Each feature = more hand-written code.


LCEL is the LangChain Expression Language for composing and running chains.


| | Plain function | LCEL chain (`prompt \| model \| parser`) |
|--|--|--|
| Reads in run order | sort of | yes, left→right |
| `batch()` | write a loop | free |
| `stream()` | rewrite with `yield` | free |
| `async` | rewrite with `async/await` | free (`ainvoke`/`astream`) |
| Swap a piece | edit function body | change one term in the pipe |
| Tracing (LangSmith) | add manually | automatic per step |

**Takeaway:** LCEL isn't ceremony — it makes every chain uniformly support batching, streaming, async, and tracing **without rewriting**. That uniformity is exactly what LangGraph relies on when it runs your chains as graph nodes.

## 9. Swapping the model — the payoff of the uniform interface

Because the model is just a Runnable, you change **one line** to swap it; prompt, parser, and `|` wiring stay identical.

**Other Gemini models** — trade speed vs quality:
```python
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # fast, cheap
model = ChatGoogleGenerativeAI(model="gemini-2.5-pro",   temperature=0)   # stronger
```

**A different provider** (e.g. AWS Bedrock later in Module 2) — same chain shape:
```python
from langchain_aws import ChatBedrock
model = ChatBedrock(model_id="anthropic.claude-3-5-sonnet-20240620-v1:0")
chain = prompt | model | parser   # unchanged
```

**Testing with no API call** — a fake model that replays canned text (deterministic, free, offline). Useful in unit tests so CI doesn't hit the real LLM:


In [14]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

fake = FakeListChatModel(responses=["canned reply for tests"])
test_chain = prompt | fake | parser          # same shape, no network call
print(test_chain.invoke({"role": "x", "question": "anything"}))


canned reply for tests


## 10. Recap

| Concept | What it is (plain English) | Why it's used |
|---------|----------------------------|---------------|
| **Runnable** | a component that speaks `invoke`/`batch`/`stream` | one uniform interface → any piece plugs into any piece |
| **LCEL `\|`** | the pipe that chains Runnables | compose steps in run order; no glue code |
| **`ChatPromptTemplate`** | template with `{placeholders}` + roles | separate fixed template from runtime values; reusable |
| **`ChatGoogleGenerativeAI`** | Gemini wrapped as a Runnable | the actual LLM call; messages → `AIMessage` |
| **`StrOutputParser`** | `AIMessage` → `str` | get clean text out, not a message object |
| **`RunnableLambda`** | wrap any function as a step | insert custom pre/post logic |
| **`RunnablePassthrough`** | pass input through / add fields | feed multiple keys to a prompt |

### Done today
- [x] Understood chains, Runnables, LCEL
- [x] Built `prompt | model | parser` with Google Gemini
- [x] Ran it locally (invoke / batch / stream)

**Next — Day 02:** LangGraph fundamentals — `StateGraph`, nodes, edges, and *why* a graph beats a plain LCEL chain once agents need loops, branching, and memory.

📚 Resources: python.langchain.com · LangChain v0.3 migration guide
🐍 Python (30 min) today: Type Hints & Pydantic v2 — see `Python/Day01.ipynb`
